I2 portable reader candidate. New outputs are isolated under runs/. Constructed examples demonstrate methods; fixed filing facts/figures are historical teaching inputs. This does not reproduce a market strategy.

In [ ]:
from pathlib import Path
import os, sys
from uuid import uuid4
ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir())
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "code"))
os.chdir(ROOT)
os.environ["MPLBACKEND"] = "Agg"
RUN_DIR = ROOT / "runs" / ("ch04_calculus_" + uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
print("Default offline teaching; current empirical French execution SKIPPED (permissions unresolved).")
print("Run directory:", RUN_DIR)


# Chapter 4: derivatives and local approximations

All values are constructed teaching examples. For a two-year zero-coupon cash flow of 100, \(P(y)=100/(1+y)^2\). At \(y=0.05\), compare the exact derivatives with central finite differences. The shared numerical functions check finite real inputs and positive scalar step sizes.

In [ ]:
from pathlib import Path
import sys
ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "src" / "quant_course").is_dir())
sys.path.insert(0, str(ROOT / "src"))
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from quant_course.math_tools import finite_difference_gradient, finite_difference_hessian

def zero_coupon_price(point):
    return 100.0 / (1.0 + point[0]) ** 2

y = 0.05
price = zero_coupon_price(np.array([y]))
first_exact = -200.0 / (1.0 + y) ** 3
second_exact = 600.0 / (1.0 + y) ** 4
first_numeric = finite_difference_gradient(zero_coupon_price, [y], step=1e-4)[0]
second_numeric = finite_difference_hessian(zero_coupon_price, [y], step=1e-3)[0, 0]
assert np.isclose(price, 90.7029478458)
assert np.isclose(first_numeric, first_exact, rtol=1e-7)
assert np.isclose(second_numeric, second_exact, rtol=2e-6)
pd.Series({"price": price, "P_prime_exact": first_exact, "P_prime_central": first_numeric,
           "P_second_exact": second_exact, "P_second_central": second_numeric})


## Two-dimensional quadratic loss

Use \(f(w)=	frac12 w^	op\Sigma w-\mu^	op w\), with the symmetric covariance-like matrix and mean vector shown below. Its exact gradient is \(\Sigma w-\mu\); its Hessian is \(\Sigma\). The numerical estimates are checked at \(w=(0.6,0.4)\).

In [ ]:
SIGMA = np.array([[0.04, 0.01], [0.01, 0.09]])
MU = np.array([0.05, 0.08])
def quadratic_loss(point):
    return float(0.5 * point @ SIGMA @ point - MU @ point)

w = np.array([0.6, 0.4])
gradient_exact = SIGMA @ w - MU
hessian_exact = SIGMA.copy()
gradient_numeric = finite_difference_gradient(quadratic_loss, w)
hessian_numeric = finite_difference_hessian(quadratic_loss, w)
assert np.allclose(gradient_numeric, gradient_exact, atol=1e-10)
assert np.allclose(hessian_numeric, hessian_exact, atol=1e-7)
print(f"f(w)={quadratic_loss(w):.8f}")
print("gradient exact:", gradient_exact)
print("gradient numeric:", gradient_numeric)
print("Hessian exact:\n", hessian_exact)
print("Hessian numeric:\n", hessian_numeric)


## Taylor error as the step changes

The first-order expansion omits quadratic curvature, so its error grows with the square of the step size along a fixed direction. For this exact quadratic, the second-order expansion recovers the function apart from floating-point roundoff. The plot keeps both series visible to show their different scales.

In [ ]:
direction = np.array([1.0, -0.5])
base_value = quadratic_loss(w)
steps = np.geomspace(1e-5, 0.25, 80)
first_errors, second_residuals = [], []
for h in steps:
    delta = h * direction
    actual = quadratic_loss(w + delta)
    first = base_value + gradient_exact @ delta
    second = first + 0.5 * delta @ hessian_exact @ delta
    first_errors.append(abs(actual - first))
    second_residuals.append(abs(actual - second))

fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.loglog(steps, first_errors, label="First-order Taylor error", color="#2878B5")
ax.loglog(steps, second_residuals, label="Second-order Taylor residual", color="#D9534F")
ax.loglog(steps, 0.5 * (direction @ SIGMA @ direction) * steps**2,
          linestyle="--", color="#555555", label="Quadratic O(h²) reference")
ax.set_xlabel("Step size h along direction [1, -0.5]")
ax.set_ylabel("Absolute approximation error")
ax.set_title("Taylor approximation error for the constructed quadratic loss")
ax.grid(True, which="both", alpha=0.25)
ax.legend()
fig.tight_layout()
output = RUN_DIR / "figures" / "a2" / "ch04_taylor_error.png"
output.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(output, dpi=160)
plt.close(fig)
output
